# Fine-tuning LoRA/QLoRA — Text2Cypher em portugues (fluxo assistencial do SUS)

Ajusta um modelo pequeno e aberto (Qwen2.5-1.5B-Instruct) com **LoRA** para traduzir
perguntas em portugues para consultas Cypher sobre o grafo do SUS. Roda na **GPU gratuita
do Kaggle** (T4) em bf16 (o modelo cabe sem quantizacao). O caminho **QLoRA 4-bit**, para
modelos maiores, esta documentado na Secao 4. O dataset vem da destilacao do Gemini
(`src/gerar_dataset.py`).

## Antes de rodar
1. Kaggle: **Settings -> Accelerator -> GPU T4 x1**.
2. Faca upload do `dataset_text2cypher.jsonl` (gerado por `src/gerar_dataset.py`) como
   **Kaggle Dataset** e adicione ao notebook. Ajuste `DATASET_PATH` na celula de config.
3. Rode as celulas em ordem. Ao final, baixe a pasta `adapter/` (o adapter LoRA) e
   aponte `ADAPTER_DIR` no `.env` do repo para servir o modelo localmente.

O objetivo e comprovar o fluxo LoRA/QLoRA: destilar do professor -> treinar o aluno ->
avaliar com a mesma metrica do `src/avaliar.py` -> comparar com a API.

## 1. Dependencias (instala no ambiente do Kaggle)

Detecta a GPU sorteada (T4 ou P100) SEM importar torch e, se for P100 (Pascal,
sm_60, que o torch da imagem atual do Kaggle nao suporta), reinstala um torch
compativel com Pascal. Depois fixa a stack de fine-tuning. peft em 0.14.0 (versoes
novas exigem torchao>0.16, ausente na imagem). Sem bitsandbytes: o modelo e
pequeno (1.5B) e treina em bf16.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

# IMPORTANTE: nao importar torch nesta celula (para poder troca-lo antes do 1o import)
smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpu = smi.stdout.strip()
print("GPU sorteada:", gpu or "(nenhuma)")

if "P100" in gpu:
    print("P100 (sm_60): instalando torch 2.4.1 com suporte a Pascal...")
    pip("torch==2.4.1", "torchvision==0.19.1", "--index-url",
        "https://download.pytorch.org/whl/cu121")

# stack de fine-tuning (nao toca no torch)
pip("transformers==4.48.0", "peft==0.14.0", "trl==0.13.0",
    "accelerate==1.2.1", "datasets==3.2.0")
print("dependencias instaladas")

## 2. Configuracao

In [ ]:
import os, json, re, glob, torch
from pathlib import Path

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"   # cabe na T4 em 4-bit

# auto-descobre o dataset onde quer que o Kaggle o monte (robusto ao nome da pasta)
if os.path.exists("/kaggle/input"):
    print("inputs montados:", os.listdir("/kaggle/input"))
_cand = glob.glob("/kaggle/input/**/dataset_text2cypher.jsonl", recursive=True)
DATASET_PATH = _cand[0] if _cand else "/kaggle/input/text2cypher-sus/dataset_text2cypher.jsonl"
print("DATASET_PATH:", DATASET_PATH, "| existe:", os.path.exists(DATASET_PATH))
OUT_DIR      = "/kaggle/working/adapter"
MAX_LEN      = 1024
VAL_FRAC     = 0.1
SEED         = 42

ESQUEMA = """Grafo Neo4j do fluxo assistencial do SUS (recorte oncologico, Sao Paulo).
Nos: (:RegiaoSaude {codigo,nome,drs,polo_oncologico}); (:Municipio {codigo,nome,populacao,idhm_renda,dist_polo_km}); (:Estabelecimento {cnes,nome,tipo}); (:Procedimento {codigo,nome,grupo,linha,complexidade,cid_grupo}).
Relacoes: (:Municipio)-[:PERTENCE_A]->(:RegiaoSaude); (:RegiaoSaude)-[:FLUXO {ano,complexidade,linha,volume,deslocou}]->(:RegiaoSaude) (origem=residencia, destino=atendimento, deslocou=true se origem<>destino); (:Estabelecimento)-[:LOCALIZADO_EM]->(:Municipio); (:Estabelecimento)-[:REALIZA {ano,volume}]->(:Procedimento).
complexidade em {basica,media,alta}; linha em {radioterapia,quimioterapia,diagnostico}."""

SYSTEM = (
    "Voce traduz perguntas em portugues para consultas Cypher sobre um grafo Neo4j. "
    "Use SOMENTE os rotulos, relacoes e propriedades do esquema. Responda com UMA "
    "consulta Cypher, sem explicacao.\n\nESQUEMA:\n" + ESQUEMA
)
# bf16 so em Ampere+ (sm_80+). T4/P100 usam fp16.
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE   = torch.bfloat16 if BF16_OK else torch.float16
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| bf16:", BF16_OK, "| dtype:", DTYPE)

## 3. Dataset -> formato de chat (train/val)

In [ ]:
from datasets import Dataset

registros = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if linha:
            registros.append(json.loads(linha))
print("pares carregados:", len(registros))

def to_messages(r):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": r["pergunta"]},
        {"role": "assistant", "content": r["cypher"]},
    ]}

ds = Dataset.from_list([to_messages(r) for r in registros]).shuffle(seed=SEED)
n_val = max(1, int(len(ds) * VAL_FRAC))
val_ds, train_ds = ds.select(range(n_val)), ds.select(range(n_val, len(ds)))
print("treino:", len(train_ds), "| val:", len(val_ds))

## 4. Modelo base (LoRA em bf16)

Qwen2.5-1.5B cabe folgado na T4 (16 GB) em bf16, entao treinamos LoRA sem
quantizacao. Para **QLoRA 4-bit** (util em modelos maiores, ex. 7B+), instale
`bitsandbytes` compativel com a CUDA do ambiente e passe um `BitsAndBytesConfig`
com `load_in_4bit=True` no `from_pretrained` (o restante do fluxo LoRA e igual).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=DTYPE, device_map="auto"
)
model.config.use_cache = False

## 5. Adapter LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

model.gradient_checkpointing_enable()
model.enable_input_require_grads()   # necessario com gradient checkpointing
lora = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Treino (SFT sobre a resposta)

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir="/kaggle/working/ckpt",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    bf16=BF16_OK,
    fp16=not BF16_OK,
    max_seq_length=MAX_LEN,
    packing=False,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, processing_class=tok)
trainer.train()

## 7. Salva o adapter

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("adapter salvo em", OUT_DIR, "-> baixe esta pasta e aponte ADAPTER_DIR no .env")

## 8. Avaliacao rapida no conjunto de validacao

Mesma logica do `src/avaliar.py`: normaliza e compara o Cypher gerado com a referencia
(acerto exato normalizado). A validacao de execucao contra o Neo4j fica no `src/avaliar.py`,
com o grafo carregado. Aqui medimos o quanto o aluno reproduz o professor.

In [ ]:
def exato(c):
    return re.sub(r"\s+", " ", c.strip().lower()).rstrip(";").strip()

def estrutural(c):
    # match justo: ignora nomes de variavel (o/r/d/f), mantendo rotulos, relacoes,
    # propriedades, palavras-chave e literais. Duas queries equivalentes ficam iguais.
    c = re.sub(r"//.*", "", c).strip().rstrip(";").lower()
    c = re.sub(r"\(\s*[a-z_]\w*\s*:", "(:", c)      # (o:label) -> (:label)
    c = re.sub(r"\[\s*[a-z_]\w*\s*:", "[:", c)      # [f:rel]   -> [:rel]
    c = re.sub(r"\b[a-z_]\w*\.", ".", c)             # o.nome    -> .nome
    c = re.sub(r"\bas\s+[a-z_]\w*", "as x", c)       # AS regiao -> AS x
    return re.sub(r"\s+", " ", c).strip()

def gera(pergunta, max_new=220):
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":pergunta}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return txt.strip().strip("`").replace("cypher\n", "").strip()

n_exato = n_estrut = 0
for ex in val_ds:
    perg = ex["messages"][1]["content"]
    ref  = ex["messages"][2]["content"]
    pred = gera(perg)
    e  = exato(pred) == exato(ref)
    es = estrutural(pred) == estrutural(ref)
    n_exato += e; n_estrut += es
    print(("EX " if e else ("ST " if es else "-- ")), perg[:58])
    if not es:
        print("    ref :", ref[:90])
        print("    pred:", pred[:90])
N = len(val_ds)
print(f"\nAcerto EXATO:      {n_exato}/{N} = {n_exato/N:.1%}")
print(f"Acerto ESTRUTURAL: {n_estrut}/{N} = {n_estrut/N:.1%}  (ignora nomes de variavel)")